# Composite KGE validation figure

This notebook consolidates the source code for `figures/kge_boxplot.png`, `figures/kge_boxplot_bc.png`, and `figures/KGE_diff_vs_KS.png`, originally produced in `two_stage_validation.ipynb`.

The composite uses three rows: the original-WRF KGE comparison, the bias-corrected-WRF KGE comparison, and the three KS–KGE-difference relationships. Rows 1 and 2 show the complete 14-station distributions for all four experiments as four boxplots per month. Typography and styling are shared across all panels.

In [1]:
from pathlib import Path
import contextlib
import io
import time

import numpy as np
import scipy.stats as stats
from scipy.stats import kstest
from sklearn.linear_model import LinearRegression
from pykrige.ok import OrdinaryKriging

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

from utils import gp_interpolator, kge

# One typography specification is used by every panel.
FONT_FAMILY = ['Myriad Pro', 'DejaVu Sans']
FONT_SIZE = 11
TICK_SIZE = 10
LEGEND_SIZE = 11
LINE_WIDTH = 1.0

mpl.rcParams.update({
    'font.family': FONT_FAMILY,
    'font.size': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': FONT_SIZE,
    'xtick.labelsize': TICK_SIZE,
    'ytick.labelsize': TICK_SIZE,
    'legend.fontsize': LEGEND_SIZE,
    'axes.linewidth': LINE_WIDTH,
    'figure.dpi': 150,
})

IPCC_BLUE = (112 / 255, 160 / 255, 205 / 255, 1)
IPCC_ORANGE = (196 / 255, 121 / 255, 0 / 255, 1)
KRIGING_COLOR = '#59A14F'
RAW_COLOR = '#737373'
MONTH_NAMES = np.array([
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec',
])

FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)
CACHE_FILE = Path('intermediate/kge_composite_validation_metrics.npz')
CACHE_FILE.parent.mkdir(exist_ok=True)

## Data preparation

This follows the data-selection logic in `two_stage_validation.ipynb`. Only the WRF grid cells corresponding to the 14 stations are retained.

In [2]:
rain_obs = np.loadtxt('data/sta_monthly.csv')
sim_sel = np.loadtxt('data/wrf_loc.csv')
sim_idx = sim_sel[:, :2].astype(int)
n_cols = 160
flat_station_idx = sim_idx[:, 0] * n_cols + sim_idx[:, 1]

sta_loc = np.genfromtxt('data/sta_lookup_new.csv', delimiter=',')[:, 2:]
kge_kriging = np.loadtxt('kriging_kge.csv')

N_MONTHS = 12
N_STATIONS = rain_obs.shape[1]
N_YEARS = rain_obs.shape[0] // N_MONTHS

assert N_STATIONS == sim_idx.shape[0] == sta_loc.shape[0]
assert kge_kriging.shape == (N_MONTHS, N_STATIONS)
print(f'Loaded {N_YEARS} years and {N_STATIONS} stations.')

Loaded 40 years and 14 stations.


## Leave-one-station-out validation

The computation is the same as the original validation routine, restricted to the metrics required by the composite figure. Internal station-noise iteration messages are suppressed; one timing line is printed per month.

In [3]:
METRIC_NAMES = (
    'kge_raw_wrf',
    'kge_gp_stage1',
    'kge_gp_stage2',
    'ks_statistic',
)


def run_two_stage_validation_metrics(rain_obs, wrf_sta, sta_loc, bias_correct=False):
    wrf_data = wrf_sta.copy()
    if bias_correct:
        wrf_data = np.zeros_like(wrf_sta)
        for month in range(N_MONTHS):
            for station in range(N_STATIONS):
                sim = wrf_sta[month::N_MONTHS, station]
                obs = rain_obs[month::N_MONTHS, station]
                linear_map = stats.linregress(np.sort(sim), np.sort(obs))
                wrf_data[month::N_MONTHS, station] = (
                    linear_map.slope * sim + linear_map.intercept
                )

    results = {
        name: np.zeros((N_MONTHS, N_STATIONS))
        for name in METRIC_NAMES
    }

    label = 'bias-corrected' if bias_correct else 'original'
    for month in range(N_MONTHS):
        start_time = time.time()
        obs_month = rain_obs[month::N_MONTHS, :]
        sim_month = wrf_data[month::N_MONTHS, :]

        for station in range(N_STATIONS):
            training_mask = np.ones(N_STATIONS, dtype=bool)
            training_mask[station] = False

            train_obs = obs_month[:, training_mask]
            train_sim = sim_month[:, training_mask]
            train_loc = sta_loc[training_mask, :]
            target_sim = sim_month[:, station][:, None]
            target_obs = obs_month[:, station][:, None]

            gp = gp_interpolator(P=N_STATIONS - 1)
            gp.read_rainfall(obs=train_obs, sim=train_sim)
            with contextlib.redirect_stdout(io.StringIO()):
                gp.sn_converge()

            stage1_pred, _ = gp.predict(target_sim)
            stage1_train, _ = gp.predict(train_sim)

            residuals = train_obs - stage1_train.T
            stage2_pred = np.zeros_like(stage1_pred)
            for time_idx, residual in enumerate(residuals):
                ordinary_kriging = OrdinaryKriging(
                    train_loc[:, 0],
                    train_loc[:, 1],
                    residual,
                    variogram_model='gaussian',
                )
                kriged_residual, _ = ordinary_kriging.execute(
                    'points', sta_loc[station, 0], sta_loc[station, 1]
                )
                stage2_pred[time_idx] = (
                    stage1_pred[time_idx] + kriged_residual
                )

            stage1_pred = np.maximum(stage1_pred, 0)
            stage2_pred = np.maximum(stage2_pred, 0)
            observed = target_obs.squeeze()
            simulated = target_sim.squeeze()

            results['kge_raw_wrf'][month, station] = kge(observed, simulated)
            results['kge_gp_stage1'][month, station] = kge(
                observed, stage1_pred.squeeze()
            )
            results['kge_gp_stage2'][month, station] = kge(
                observed, stage2_pred.squeeze()
            )

            if not bias_correct:
                z_values = stats.norm.cdf(
                    observed,
                    loc=np.mean(simulated),
                    scale=np.std(simulated, ddof=1),
                )
                results['ks_statistic'][month, station] = kstest(
                    np.sort(z_values), 'uniform'
                ).statistic

        elapsed = time.time() - start_time
        print(f'{label}: month {month + 1:02d}/{N_MONTHS} ({elapsed:.2f} s)')

    return results

In [4]:
# Set to True to deliberately rerun the full validation.
FORCE_RECOMPUTE = False

if CACHE_FILE.exists() and not FORCE_RECOMPUTE:
    with np.load(CACHE_FILE) as cached:
        results_raw = {
            name: cached[f'raw_{name}'] for name in METRIC_NAMES
        }
        results_bc = {
            name: cached[f'bc_{name}'] for name in METRIC_NAMES
        }
    print(f'Loaded cached validation metrics from {CACHE_FILE}.')
else:
    rain_sim_flatten = np.loadtxt('data/wrf_monthly.csv')
    wrf_sta = rain_sim_flatten[:, flat_station_idx]
    del rain_sim_flatten
    assert rain_obs.shape == wrf_sta.shape

    results_raw = run_two_stage_validation_metrics(
        rain_obs, wrf_sta, sta_loc, bias_correct=False
    )
    results_bc = run_two_stage_validation_metrics(
        rain_obs, wrf_sta, sta_loc, bias_correct=True
    )
    np.savez_compressed(
        CACHE_FILE,
        **{f'raw_{name}': results_raw[name] for name in METRIC_NAMES},
        **{f'bc_{name}': results_bc[name] for name in METRIC_NAMES},
    )
    print(f'Saved validation metrics to {CACHE_FILE}.')

for result_set in (results_raw, results_bc):
    for name in METRIC_NAMES:
        assert result_set[name].shape == (N_MONTHS, N_STATIONS)

Loaded cached validation metrics from intermediate/kge_composite_validation_metrics.npz.


## Shared plotting functions

The boxplot and KS-regression code below is refactored from the three original figure cells so the same formatting rules apply everywhere.

In [5]:
def plot_kge_row(ax, results, panel_title):
    positions = np.arange(N_MONTHS, dtype=float)
    method_specs = [
        ('Raw', results['kge_raw_wrf'], RAW_COLOR),
        ('Stage 1', results['kge_gp_stage1'], IPCC_ORANGE),
        ('Stage 1 + 2', results['kge_gp_stage2'], IPCC_BLUE),
        ('Ordinary kriging', kge_kriging, KRIGING_COLOR),
    ]
    offsets = np.array([-0.30, -0.10, 0.10, 0.30])

    for offset, (_, values, color) in zip(offsets, method_specs):
        monthly_values = [
            values[month, np.isfinite(values[month])]
            for month in range(N_MONTHS)
        ]
        ax.boxplot(
            monthly_values,
            positions=positions + offset,
            widths=0.16,
            patch_artist=True,
            manage_ticks=False,
            whis=1.5,
            showfliers=True,
            boxprops={
                'facecolor': color, 'edgecolor': 'black',
                'linewidth': LINE_WIDTH,
            },
            medianprops={'color': 'black', 'linewidth': 1.3},
            whiskerprops={'color': 'black', 'linewidth': LINE_WIDTH},
            capprops={'color': 'black', 'linewidth': LINE_WIDTH},
            flierprops={
                'marker': 'o', 'markerfacecolor': 'none',
                'markeredgecolor': color, 'markersize': 3.5,
                'linestyle': 'none',
            },
        )

    ax.set_xlim(-0.6, N_MONTHS - 0.4)
    ax.set_ylim(-0.5, 1.0)
    ax.set_xticks(positions, MONTH_NAMES)
    ax.set_yticks([-0.5, -0.41, 0, 0.5, 1.0])
    ax.set_yticklabels(['-0.5', '-0.41', '0', '0.5', '1'])
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
    ax.grid(which='both', axis='y', linestyle='--', color='0.72', linewidth=0.7)
    ax.axhline(-0.41, color='black', linestyle='--', linewidth=LINE_WIDTH)
    ax.text(
        0.99, -0.395, 'Baseline KGE score using climatology',
        transform=ax.get_yaxis_transform(),
        ha='right', va='bottom', fontsize=TICK_SIZE,
        bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.85, 'pad': 1.5},
    )
    ax.set_ylabel('KGE')
    ax.set_title(panel_title, loc='left', pad=5)
    ax.tick_params(which='both', labelsize=TICK_SIZE)


def fit_power_curve(x, y, exponent):
    keep = y >= -0.2
    model = LinearRegression(fit_intercept=False).fit(
        x[keep, None] ** exponent, y[keep]
    )
    coefficient = model.coef_[0]
    fitted = model.predict(x[keep, None] ** exponent)
    rss = np.sum((y[keep] - fitted) ** 2)
    tss = np.sum((y[keep] - np.mean(y[keep])) ** 2)
    r_squared = 1 - rss / tss
    return model, coefficient, r_squared


def plot_ks_panel(ax, x, y, exponent, panel_label, description):
    ax.axhline(
        0, color='0.45', linestyle=':', linewidth=1.0, zorder=0
    )
    ax.scatter(x, y, s=14, marker='o', color='#1f77b4', alpha=0.95)
    model, coefficient, r_squared = fit_power_curve(x, y, exponent)
    x_line = np.linspace(np.min(x), np.max(x), 100)
    y_line = model.predict(x_line[:, None] ** exponent)
    ax.plot(x_line, y_line, color='black', linestyle='--', linewidth=1.3)

    ax.set_ylim(-0.2, 0.6)
    ax.set_title(
        f'({panel_label}) {description}',
        pad=5,
    )
    formula = (
        rf'$Y = {coefficient:.2f} \cdot X^{{{exponent:.1f}}}$'
        '\n'
        rf'$R^2 = {r_squared:.2f}$'
    )
    ax.text(0.04, 0.95, formula, transform=ax.transAxes, va='top')
    ax.tick_params(which='both', labelsize=TICK_SIZE)
    return coefficient, r_squared

## Three-row composite

Rows 1 and 2 span the full figure width. Row 3 preserves the original three side-by-side KS panels.

In [6]:
fig = plt.figure(figsize=(9.2, 9.2))
grid = fig.add_gridspec(
    3, 3,
    height_ratios=[1.0, 1.0, 1.12],
    hspace=0.25, wspace=0.20,
)

ax_raw = fig.add_subplot(grid[0, :])
ax_bc = fig.add_subplot(grid[1, :], sharex=ax_raw, sharey=ax_raw)
ax_ks = [
    fig.add_subplot(grid[2, 0]),
    fig.add_subplot(grid[2, 1]),
    fig.add_subplot(grid[2, 2]),
]

plot_kge_row(ax_raw, results_raw, '(a) Original WRF prior')
plot_kge_row(ax_bc, results_bc, '(b) Idealized station-wise bias-corrected WRF prior')

legend_handles = [
    Patch(facecolor=RAW_COLOR, edgecolor='black', label='Unconditioned WRF'),
    Patch(facecolor=IPCC_ORANGE, edgecolor='black', label='Stage 1'),
    Patch(facecolor=IPCC_BLUE, edgecolor='black', label='Stage 1 + 2'),
    Patch(facecolor=KRIGING_COLOR, edgecolor='black', label='Ordinary Kriging'),
]
fig.legend(
    handles=legend_handles, loc='upper center', ncol=4,
    bbox_to_anchor=(0.64, 0.98), frameon=False,
)

ks_values = results_raw['ks_statistic'].ravel()
kge_differences = [
    results_bc['kge_gp_stage2'].ravel() - results_raw['kge_gp_stage1'].ravel(),
    results_bc['kge_gp_stage1'].ravel() - results_raw['kge_gp_stage1'].ravel(),
    results_raw['kge_gp_stage2'].ravel() - results_raw['kge_gp_stage1'].ravel(),
]
exponents = [1.8, 2.4, 2.5]
descriptions = [
    'BC Stage 1+2 − Original Stage 1',
    'BC Stage 1 − Original Stage 1',
    'Original Stage 1+2 − Original Stage 1',
]

fit_summary = []
for axis, difference, exponent, label, description in zip(
    ax_ks, kge_differences, exponents, ['c', 'd', 'e'], descriptions
):
    fit_summary.append(
        plot_ks_panel(
            axis, ks_values, difference, exponent, label, description
        )
    )

ax_ks[0].set_ylabel('KGE Difference')
ax_ks[1].set_xlabel('K-S Statistic')
ax_ks[1].tick_params(labelleft=False)
ax_ks[2].tick_params(labelleft=False)

fig.subplots_adjust(left=0.09, right=0.985, bottom=0.065, top=0.94)
output_path = FIGURE_DIR / 'kge_three_row_composite.png'
fig.savefig(output_path, dpi=600, bbox_inches='tight')
print(f'Saved {output_path}')
plt.show()

Saved figures/kge_three_row_composite.png


/var/folders/wy/_8r9h_1562q_4cj3h4l5ht0m0000gn/T/ipykernel_7276/1568643597.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
